# Overnight Stock Bars Example

This example uses `exchange="OVERNIGHT"` for the underlying stock market-data contract and stock execution instrument. It also sets `useRTH=False` and `outside_rth=True`.

IBKR overnight availability depends on the symbol, account permissions, and data subscriptions.

In [1]:
from __future__ import annotations

import copy
import sys
from pathlib import Path

from ib_async import IB, util

# Required for sync ib.connect(...) inside Jupyter/IPython kernels.
util.startLoop()

repo_root = Path.cwd().resolve()
if repo_root.name == "examples":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import DEFAULT_LIVE_CONFIG, LiveTradingConfig
from execution import (
    active_orders,
    filled_orders,
    recent_fills,
    strategy_open_trades,
    print_order_snapshot,
    build_execution_instrument,
    request_underlying_realtime_bars,
)
from orders import IBKRLimitOrderRouter
from strategies.live_mean_reversion import LiveMeanReversion


In [2]:
SYMBOL = "AMD"
IB_HOST = "127.0.0.1"
IB_PORT = 4002
IB_CLIENT_ID = 113

raw = copy.deepcopy(DEFAULT_LIVE_CONFIG)
raw["symbol"] = SYMBOL
raw["model"]["enabled"] = False
raw["features"]["window"] = 1
raw["features"]["min_bars"] = 2
raw["strategy"]["entry_strategy"] = {"name": "always_true"}
raw["strategy"]["no_trade_first_minutes"] = -1000
raw["strategy"]["no_new_entries_last_minutes"] = -1000
raw["strategy"]["entry_price_limits"] = {
    "first_entry": {"max_stock_price": 550.0, "max_buy_price": 550.0},
}
raw["double_down"]["price_rules"] = []
raw["strategy"]["latest_trade_price_rules"] = {
    "enabled": True,
    "first_entry": [
        {"basis": "underlying", "mode": "pct", "max_change": -0.01},
        {"basis": "execution", "mode": "pct", "max_change": -0.01},
    ],
    "double_down": [
        {"basis": "execution", "mode": "abs", "max_change": -5.00},
    ],
}
raw["execution"]["market_data"] = {
    "exchange": "OVERNIGHT",
    "barSize": 5,
    "whatToShow": "TRADES",
    "useRTH": False,
}
raw["execution"]["outside_rth"] = True
raw["execution"]["instrument"] = {
    "type": "stock",
    "exchange": "OVERNIGHT",
    "currency": "USD",
    "limit_entry_offset_pct": 0.0005,
    "limit_exit_offset_pct": 0.0005,
}

config = LiveTradingConfig.from_dict(raw)
config.execution, config.strategy["latest_trade_price_rules"]


({'tif': 'DAY',
  'outside_rth': True,
  'market_data': {'exchange': 'OVERNIGHT',
   'barSize': 5,
   'whatToShow': 'TRADES',
   'useRTH': False},
  'instrument': {'type': 'stock',
   'exchange': 'OVERNIGHT',
   'currency': 'USD',
   'limit_entry_offset_pct': 0.0005,
   'limit_exit_offset_pct': 0.0005}},
 {'enabled': True,
  'first_entry': [{'basis': 'underlying', 'mode': 'pct', 'max_change': -0.01},
   {'basis': 'execution', 'mode': 'pct', 'max_change': -0.01}],
  'double_down': [{'basis': 'execution', 'mode': 'abs', 'max_change': -5.0}]})

In [3]:
ib = IB()
if not ib.isConnected():
    ib.connect(IB_HOST, IB_PORT, clientId=IB_CLIENT_ID)

stock, real_time_bars = request_underlying_realtime_bars(
    ib=ib,
    symbol=config.symbol,
    market_data_cfg=config.execution["market_data"],
)

execution_instrument = build_execution_instrument(ib, config.execution, config.symbol)
order_router = IBKRLimitOrderRouter(ib=ib, contract=stock)
algo = LiveMeanReversion(config=config, order_router=order_router, execution_instrument=execution_instrument)

print("Market data contract:", stock)
print("useRTH:", config.execution["market_data"]["useRTH"])
print("outside_rth orders:", config.execution["outside_rth"])


[LOAD OPEN TRADES] empty/corrupt file ignored: META_open_trades.csv
Market data contract: Stock(conId=4391, symbol='AMD', exchange='OVERNIGHT', primaryExchange='NASDAQ', currency='USD', localSymbol='AMD', tradingClass='NMS')
useRTH: False
outside_rth orders: True


In [17]:
# Attach when ready.
real_time_bars.updateEvent += algo.on_bar

# Detach before rerunning setup.
#real_time_bars.updateEvent -= algo.on_bar


## Status Checks

Use these cells to see whether the strategy is waiting for bars, blocked by config, has active orders, or has fills.


In [26]:
print_order_snapshot(ib, algo)
print("completed_signal_bars:", len(algo.signal_bars))
print("current_partial_bar:", algo.signal_bar_builder.current_bar)
print("last_signal:", algo.last_signal)


In [ ]:
active_orders(ib)


Exception in callback _SelectorSocketTransport._read_ready()
handle: <Handle _SelectorSocketTransport._read_ready()>
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x104eb7500> is already entered
Exception in callback _SelectorSocketTransport._read_ready()
handle: <Handle _SelectorSocketTransport._read_ready()>
Traceback (most recent call last):
  File "/opt/miniconda3/lib/python3.13/asyncio/events.py", line 89, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x104eb7500> is already entered
Exception in callback _SelectorSocketTransport._read_ready()
handle: <Handle _SelectorSocketTransport._read_ready()>
Traceback (most recent 

In [23]:
filled_orders(ib)


""


In [13]:
recent_fills(ib)


""


In [11]:
strategy_open_trades(algo)


""


In [12]:
features = algo.calculate_features()
if features is None:
    print("features not ready yet; need more completed signal bars")
else:
    signal = algo.should_enter_trade(features)
    print("candidate_signal:", signal)


features not ready yet; need more completed signal bars
